# SafeWatch — 차량 검출 파인튜닝 (단일 클래스)데이터셋의 25개 클래스 중 차량 4종만 `vehicle` 단일 클래스로 통합하여 학습한다.**실행**: 1 → 2 → (3) → 4 → 5 → 6 순서로. 2번에서 백업이 복원되면 3번은 건너뛴다.**세션이 끊기면**: 1 → 2 실행 후, 학습 중이었으면 맨 아래 재개 셀로.

## 1. 환경 준비마운트에서 `credential propagation was unsuccessful` 오류가 나면 런타임 재시작 후 다시 실행한다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

from pathlib import Path
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

# ── 경로 ──────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/SafeWatch')
DRIVE_DATA = DRIVE_ROOT / '도로주행영상'
TRAIN_DIR  = DRIVE_DATA / 'Training'
VALID_DIR  = DRIVE_DATA / 'Validation'
RUNS_DIR   = DRIVE_ROOT / 'runs'

OUT    = Path('/content/yolo_vehicle')      # 최종 학습 데이터
RAW_TR = Path('/content/train_raw')
RAW_VA = Path('/content/val_raw')
BACKUP = DRIVE_ROOT / 'yolo_vehicle.tar'

# ── 설정 ──────────────────────────────────────────────
VEHICLE  = {'Vehicle_Car', 'Vehicle_Bus', 'Vehicle_Motorcycle', 'Vehicle_Unknown'}
N_TRAIN  = 6000     # 원천 zip당 이미지 수
N_VALID  = 500
RUN_NAME = 'vehicle_v1'
IMGSZ, EPOCHS, BATCH = 320, 100, 32

RUNS_DIR.mkdir(parents=True, exist_ok=True)
assert DRIVE_DATA.exists(), f'경로 없음: {DRIVE_DATA}'
print('준비 완료 →', DRIVE_DATA)

## 2. 백업 확인이전에 만들어 둔 학습 데이터가 드라이브에 있으면 복원한다. 복원되면 3번 셀은 건너뛴다.

In [ ]:
import subprocess, time

READY = False
if BACKUP.exists():
    print(f'백업 발견 [{BACKUP.stat().st_size/1e9:.2f} GB] → 복원 중')
    t0 = time.time()
    subprocess.run(['tar', '-xf', str(BACKUP), '-C', '/content'], check=True)
    READY = (OUT / 'data.yaml').exists()
    print(f'복원 {"성공" if READY else "실패"} ({time.time()-t0:.0f}s)')
else:
    print('백업 없음 → 3번 셀부터 진행')

if READY:
    DATA = OUT / 'data.yaml'
    n_tr = len(list((OUT/'images'/'train').glob('*')))
    n_va = len(list((OUT/'images'/'val').glob('*')))
    print(f'train {n_tr}장 / val {n_va}장')
    print('→ 3번 건너뛰고 4번으로')

## 3. 원본 해제 (백업이 없을 때만)`bb`(바운딩박스) 폴더만 사용한다. 라벨은 전부, 이미지는 설정한 개수만 꺼낸다.zip 종류는 이름이 아니라 **내용으로 판별**한다. 한글 파일명이 드라이브를 거치며 자모 분리되는 경우가 있어서다.

In [ ]:
import zipfile, random, shutil, time

if READY:
    print('백업에서 복원됨 — 건너뜀')
else:
    def unpack(src_dir: Path, dest: Path, n_img: int):
        shutil.rmtree(dest, ignore_errors=True)
        for z in sorted(src_dir.rglob('*.zip')):
            with zipfile.ZipFile(z) as zf:
                bb  = [x for x in zf.namelist()
                       if x.startswith('bb/') and not x.endswith('/')]
                xml = [x for x in bb if x.lower().endswith('.xml')]
                img = [x for x in bb if x.lower().endswith(('.jpg', '.png'))]

                if xml:                       # 라벨 zip → 전부
                    pick = xml
                else:                         # 원천 zip → 일부
                    random.seed(42)
                    pick = sorted(random.sample(img, min(n_img, len(img))))

                print(f'  {z.name}: bb {len(bb)}개 중 {len(pick)}개')
                for x in pick:
                    zf.extract(x, dest)

    t0 = time.time()
    print('[Training]');   unpack(TRAIN_DIR / 'bb', RAW_TR, N_TRAIN)
    print('[Validation]'); unpack(VALID_DIR,        RAW_VA, N_VALID)

    for tag, d in [('train', RAW_TR), ('val', RAW_VA)]:
        print(f'{tag}: 이미지 {len(list(d.rglob("*.jpg")))} / XML {len(list(d.rglob("*.xml")))}')
    print(f'\n완료 ({time.time()-t0:.0f}s)')
    !df -h /content | tail -1

## 4. 단일 클래스 변환XML(Pascal VOC)을 YOLO 형식으로 바꾸면서 차량 4종을 `vehicle` 하나로 통합한다.보행자·신호등·노면표시는 본 과제에 불필요하므로 제외한다.

In [ ]:
import xml.etree.ElementTree as ET
import re, shutil, yaml

if READY:
    print('백업에서 복원됨 — 건너뜀')
else:
    norm = lambda s: re.sub(r'_v\d+_\d+$', '', Path(s).stem)
    shutil.rmtree(OUT, ignore_errors=True)

    def convert(raw: Path, split: str):
        (OUT/'images'/split).mkdir(parents=True, exist_ok=True)
        (OUT/'labels'/split).mkdir(parents=True, exist_ok=True)
        xmap = {norm(x): x for x in raw.rglob('*.xml')}
        imgs = list(raw.rglob('*.jpg')) + list(raw.rglob('*.png'))
        hit  = sum(1 for p in imgs if p.stem in xmap)
        print(f'  {split}: 이미지 {len(imgs)} / 매칭 {hit}')

        ok = box = 0
        for ip in imgs:
            xp = xmap.get(ip.stem)
            if not xp: continue
            try:
                r = ET.parse(xp).getroot()
                W = float(r.findtext('size/width')); H = float(r.findtext('size/height'))
                if W <= 0 or H <= 0: continue
                lines = []
                for o in r.iter('object'):
                    if (o.findtext('name') or '').strip() not in VEHICLE: continue
                    b = o.find('bndbox')
                    x1 = max(0, min(float(b.findtext('xmin')), W))
                    y1 = max(0, min(float(b.findtext('ymin')), H))
                    x2 = max(0, min(float(b.findtext('xmax')), W))
                    y2 = max(0, min(float(b.findtext('ymax')), H))
                    if x2-x1 < 1 or y2-y1 < 1: continue
                    lines.append(f'0 {((x1+x2)/2)/W:.6f} {((y1+y2)/2)/H:.6f} '
                                 f'{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
                if not lines: continue
                shutil.copy(ip, OUT/'images'/split/ip.name)
                (OUT/'labels'/split/f'{ip.stem}.txt').write_text('\n'.join(lines))
                ok += 1; box += len(lines)
            except Exception:
                pass
        print(f'     → {ok}장 / 박스 {box}개 (장당 {box/max(ok,1):.1f})')
        return ok

    n_tr = convert(RAW_TR, 'train')
    n_va = convert(RAW_VA, 'val')
    assert n_tr > 0 and n_va > 0, '변환 결과가 비어 있음'

    DATA = OUT / 'data.yaml'
    yaml.safe_dump({'path': str(OUT), 'train': 'images/train',
                    'val': 'images/val', 'names': {0: 'vehicle'}},
                   open(DATA, 'w'), allow_unicode=True)
    print('\nDATA =', DATA)

## 5. 백업**학습 전에 반드시 실행한다.** 세션이 끊기면 `/content`가 통째로 사라지므로,여기서 백업해 두면 다음 세션은 2번 셀에서 복원만 하면 된다.

In [ ]:
import subprocess, time

if BACKUP.exists():
    print('백업 이미 있음 — 건너뜀')
else:
    t0 = time.time()
    subprocess.run(['tar', '-cf', '/content/yolo_vehicle.tar',
                    '-C', '/content', 'yolo_vehicle'], check=True)
    subprocess.run(['cp', '/content/yolo_vehicle.tar', str(BACKUP)], check=True)
    print(f'백업 완료 ({time.time()-t0:.0f}s) → {BACKUP}')

## 6. 학습체크포인트는 드라이브에 저장되므로 세션이 끊겨도 남는다.

In [ ]:
model = YOLO('yolov8n.pt')

model.train(
    data=str(DATA),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=25,
    save_period=5,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    plots=True,
)
print('학습 완료')

## 7. 평가비교 기준 — 25개 클래스로 학습했을 때의 **차량 4종 평균**은 다음과 같았다.| | P | R | mAP50 ||---|---|---|---|| 25클래스 모델의 차량 평균 | 0.602 | 0.491 | 0.500 || 25클래스 전체 평균 | 0.459 | 0.190 | 0.193 |아래 결과가 첫 번째 행보다 높아야 단일 클래스 통합이 효과가 있었다는 뜻이다.

In [ ]:
best = YOLO(str(RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'))
m = best.val(data=str(DATA), imgsz=IMGSZ)

print()
print(f'Precision : {m.box.mp:.4f}')
print(f'Recall    : {m.box.mr:.4f}')
print(f'mAP50     : {m.box.map50:.4f}')
print(f'mAP50-95  : {m.box.map:.4f}')
print()
print('참고 — 25클래스 모델 차량 평균: P 0.602 / R 0.491 / mAP50 0.500')

신뢰도 임계값에 따른 변화. 본 시스템은 복수 지표 결합 판정으로 오탐을 한 번 더 거르므로,검출 단계에서는 Recall을 조금 더 확보하는 쪽이 유리할 수 있다.

In [ ]:
print('conf     Precision  Recall   mAP50')
for c in [0.001, 0.01, 0.05, 0.1, 0.25]:
    r = best.val(data=str(DATA), imgsz=IMGSZ, conf=c, verbose=False)
    print(f'{c:<8} {r.box.mp:.3f}      {r.box.mr:.3f}    {r.box.map50:.3f}')

## 8. 먼 차량 검출률
차 바닥 높이별 Recall — 실행 입력 320 vs 640. 차선 노트북 v2 29번 셀(앞차 절반이 화면 높이 0.58보다 멂)의 후속.

In [ ]:
# 8. 먼 차량 검출률 — 차 바닥 높이별 Recall (차선 노트북 29번과 같은 높이 구간)
# 필요: 1·2번 셀 (OUT = 학습 데이터, RUNS_DIR·RUN_NAME). GPU 권장, 몇 분.
#
# 왜: 차선 노트북 29번 셀 — 앞차의 약 절반은 차 바닥이 화면 높이 0.58보다 먼 곳에 있다. 차선을 먼 곳까지 잡아도
#     차량을 못 찾으면 소용없다 → 먼 차를 얼마나 찾는지 먼저 잰다.
# 보는 것: 실행 조건(Pi 입력 320) vs 640 — 640에서 크게 오르면 원인은 해상도(학습으로는 한계), 비슷하게 낮으면 학습 문제.
# 기준: 정답 박스(val 라벨) 중 가로 가운데 70% 안(실행 detection.center_region_ratio)인 것만. IoU 0.5 이상 예측이 있으면 찾음.
#       conf 0.1 = 실행 detection.conf_threshold(기존 트랙 잇기), 0.5 = 새 트랙 시작(tracking.track_high_thresh)
import numpy as np, time
from collections import defaultdict

W8 = {'파인튜닝': RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'}   # 실행 가중치(yolov8n-safewatch)와 같은 것
IMGSZ8  = [320, 640]
CONF8   = [0.1, 0.5]
CENTER8 = 0.7
BINS8   = [0.0, 0.42, 0.50, 0.54, 0.58, 0.62, 0.66, 0.70, 0.80, 1.01]
IOU8    = 0.5

imgs8 = sorted((OUT / 'images' / 'val').glob('*'))
if not imgs8:
    raise SystemExit(f'{OUT}/images/val 비었음 — 2번 셀(복원) 먼저')


def gt8(ip):
    lp = OUT / 'labels' / 'val' / f'{ip.stem}.txt'
    if not lp.exists():
        return np.zeros((0, 4))
    a = np.loadtxt(lp, ndmin=2)[:, 1:]                   # cx cy w h (0~1)
    b = np.c_[a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2, a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2]
    cx = a[:, 0]
    return b[np.abs(cx - 0.5) <= CENTER8 / 2]


def iou8(g, p):
    x1, y1 = np.maximum(g[0], p[:, 0]), np.maximum(g[1], p[:, 1])
    x2, y2 = np.minimum(g[2], p[:, 2]), np.minimum(g[3], p[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / ((g[2] - g[0]) * (g[3] - g[1]) + (p[:, 2] - p[:, 0]) * (p[:, 3] - p[:, 1]) - inter)


GT8 = {ip: gt8(ip) for ip in imgs8}
n_gt = sum(len(v) for v in GT8.values())
print(f'val {len(imgs8)}장 · 가운데 {int(CENTER8 * 100)}% 안 정답 차량 {n_gt}대')

HIT8 = {}                                              # (모델, imgsz, conf) → [(바닥 높이, 박스 높이, 찾음)]
for name, wp in W8.items():
    model = YOLO(str(wp))
    for sz in IMGSZ8:
        t0, rows = time.time(), {c: [] for c in CONF8}
        for i in range(0, len(imgs8), 32):
            batch = imgs8[i:i + 32]
            rs = model.predict([str(p) for p in batch], imgsz=sz, conf=min(CONF8), verbose=False, device=0)
            for ip, r in zip(batch, rs):
                g = GT8[ip]
                if len(g) == 0:
                    continue
                p = r.boxes.xyxyn.cpu().numpy(); s = r.boxes.conf.cpu().numpy()
                for c in CONF8:
                    pc, used = p[s >= c], set()
                    for gb in sorted(g, key=lambda b: -(b[3] - b[1])):      # 큰(가까운) 차부터 짝짓기
                        hit = False
                        if len(pc):
                            ious = iou8(gb, pc)
                            for j in np.argsort(-ious):
                                if ious[j] < IOU8:
                                    break
                                if j not in used:
                                    used.add(j); hit = True
                                    break
                        rows[c].append((gb[3], gb[3] - gb[1], hit))
        for c in CONF8:
            HIT8[(name, sz, c)] = rows[c]
        print(f'  {name} imgsz {sz}: {time.time() - t0:.0f}초')

print('\n' + '═' * 110)
print('칸 = Recall (찾은 대수/정답 대수)     높이 = 차 바닥(화면 높이 비율, 위 = 멀리)  ·  차선 노트북 29번: 지금 offset을 재는 건 0.58 이상')
print('═' * 110)
keys = [k for k in HIT8]
print(f'  {"차 바닥 높이":<12}{"대수":>6}  ' + ''.join(f'{f"{k[0]} {k[1]} conf{k[2]}":>22}' for k in keys))
groups = [(f'{a:.2f}~{min(b, 1):.2f}', lambda y, a=a, b=b: a <= y < b) for a, b in zip(BINS8[:-1], BINS8[1:])]
groups += [('0.58 위(먼 차)', lambda y: y < 0.58), ('0.58 이상', lambda y: y >= 0.58), ('전체', lambda y: True)]
for gname, cond in groups:
    cells, n = [], 0
    for k in keys:
        rs = [r for r in HIT8[k] if cond(r[0])]
        n = len(rs)
        cells.append(f'{(sum(r[2] for r in rs) / n if n else float("nan")):.2f} ({sum(r[2] for r in rs)}/{n})' if n else '-')
    print(f'  {gname:<12}{n:>6}  ' + ''.join(f'{c:>22}' for c in cells))

print('\n박스 높이별 (화면 높이 대비, 작을수록 멀다)')
HB8 = [0, 0.02, 0.04, 0.08, 1.01]
for a, b in zip(HB8[:-1], HB8[1:]):
    cells, n = [], 0
    for k in keys:
        rs = [r for r in HIT8[k] if a <= r[1] < b]
        n = len(rs)
        cells.append(f'{sum(r[2] for r in rs) / n:.2f} ({sum(r[2] for r in rs)}/{n})' if n else '-')
    print(f'  {a:.2f}~{min(b, 1):.2f}    {n:>6}  ' + ''.join(f'{c:>22}' for c in cells))

## 9. 먼 곳 확대 검출
화면 전체 320 + 먼 영역만 잘라 320으로 한 번 더 — 640과 비교 (8번 실행 후).

In [ ]:
# 9. 먼 곳 확대 검출 — 화면 전체 320 + 먼 영역만 잘라 한 번 더 320 (같은 가중치, 학습 없음)
# 필요: 8번 셀을 이 세션에서 먼저 실행 (imgs8·GT8·iou8·HIT8 사용). GPU, 1~2분.
#
# 왜: 8번 — 먼 차(바닥 0.58 위)를 새 트랙으로 잡는 Recall이 320 입력 0.49, 640 입력 0.72. 원인이 해상도라면
#     화면 전체를 640으로 키우지 않고 먼 영역만 잘라 320으로 한 번 더 넣어도(연산 약 2배, 640은 약 4배) 비슷하게 오를 것.
# 방법: 먼 영역 = 가로 CROP_X9 · 세로 CROP_Y9 (화면 비율, 고정 — 먼 차 바닥 0.42~0.58이 들어가게).
#       잘라낸 영역 검출을 원래 좌표로 되돌리고, 잘린 경계에 걸친 박스는 버린 뒤(잘린 차), 전체 검출과 겹치지 않는(IoU 0.5 미만) 것만 더한다.
# 오탐도 센다: 가운데 70% 안 예측 중 어떤 정답과도 IoU 0.5 미만인 것 (먼 영역을 한 번 더 보면 오탐도 늘 수 있다)
import numpy as np, time

CROP_X9 = (0.25, 0.75)
CROP_Y9 = (0.30, 0.65)
EDGE9   = 0.01                                        # 잘린 경계에서 이만큼(화면 비율) 안쪽에 닿은 박스는 버림
NMS9    = 0.5

if 'HIT8' not in globals():
    raise SystemExit('8번 셀을 이 세션에서 먼저 실행하세요')


def merge9(pf, pz, thr):
    """전체 화면 검출은 그대로 두고, 먼 영역 검출 중 전체 검출과 IoU thr 이상 겹치지 않는 것만 더한다
    (전체 검출끼리는 YOLO가 이미 NMS를 했다 — 다시 거르면 붙어 있는 차 둘 중 하나가 지워질 수 있다)"""
    if len(pz) == 0 or len(pf) == 0:
        return np.r_[pf.reshape(-1, 4), pz.reshape(-1, 4)]
    new = [b for b in pz if iou8(b, pf).max() < thr]
    return np.r_[pf, np.array(new).reshape(-1, 4)]


def match9(g, p):
    """정답 g와 예측 p 짝짓기 → (정답별 찾음 여부, 가운데 70% 안 오탐 수)"""
    used, hits = set(), []
    for gb in sorted(g, key=lambda b: -(b[3] - b[1])):
        hit = False
        if len(p):
            ious = iou8(gb, p)
            for j in np.argsort(-ious):
                if ious[j] < IOU8:
                    break
                if j not in used:
                    used.add(j); hit = True
                    break
        hits.append((gb[3], gb[3] - gb[1], hit))
    cx = (p[:, 0] + p[:, 2]) / 2 if len(p) else np.zeros(0)
    fp = sum(1 for j in range(len(p)) if j not in used and abs(cx[j] - 0.5) <= CENTER8 / 2)
    return hits, fp


from PIL import Image
model9 = YOLO(str(W8['파인튜닝']))
RES9 = {(k, c): ([], 0) for k in ('320', '320+먼 영역', '640') for c in CONF8}
t0 = time.time()
for i in range(0, len(imgs8), 32):
    batch = imgs8[i:i + 32]
    full = {sz: model9.predict([str(p) for p in batch], imgsz=sz, conf=min(CONF8), verbose=False, device=0) for sz in (320, 640)}
    crops = []
    for p in batch:
        im = Image.open(p); w, h = im.size
        crops.append(im.crop((int(CROP_X9[0] * w), int(CROP_Y9[0] * h), int(CROP_X9[1] * w), int(CROP_Y9[1] * h))))
    rc = model9.predict(crops, imgsz=320, conf=min(CONF8), verbose=False, device=0)
    cw, ch = CROP_X9[1] - CROP_X9[0], CROP_Y9[1] - CROP_Y9[0]
    for j, ip in enumerate(batch):
        g = GT8[ip]
        if len(g) == 0:
            continue
        pz = rc[j].boxes.xyxyn.cpu().numpy(); sz_ = rc[j].boxes.conf.cpu().numpy()
        inner = ((pz[:, 0] > EDGE9 / cw) & (pz[:, 2] < 1 - EDGE9 / cw) & (pz[:, 1] > EDGE9 / ch) & (pz[:, 3] < 1 - EDGE9 / ch)) if len(pz) else np.zeros(0, bool)
        pz, sz_ = pz[inner], sz_[inner]
        pz = np.c_[CROP_X9[0] + pz[:, 0] * cw, CROP_Y9[0] + pz[:, 1] * ch, CROP_X9[0] + pz[:, 2] * cw, CROP_Y9[0] + pz[:, 3] * ch] if len(pz) else pz.reshape(0, 4)
        p320, s320 = full[320][j].boxes.xyxyn.cpu().numpy(), full[320][j].boxes.conf.cpu().numpy()
        p640, s640 = full[640][j].boxes.xyxyn.cpu().numpy(), full[640][j].boxes.conf.cpu().numpy()
        for c in CONF8:
            for k, q in (('320', p320[s320 >= c]), ('640', p640[s640 >= c]),
                         ('320+먼 영역', merge9(p320[s320 >= c], pz[sz_ >= c], NMS9))):
                h_, fp = match9(g, q)
                RES9[(k, c)][0].extend(h_)
                RES9[(k, c)] = (RES9[(k, c)][0], RES9[(k, c)][1] + fp)
print(f'{len(imgs8)}장 {time.time() - t0:.0f}초 · 먼 영역 가로 {CROP_X9} 세로 {CROP_Y9}')

keys9 = list(RES9)
print('\n' + '═' * 120)
print('칸 = Recall (찾은/정답)   ·   맨 아래 = 가운데 70% 안 오탐 수      conf 0.1 = 기존 트랙 잇기, 0.5 = 새 트랙 시작')
print('═' * 120)
print(f'  {"차 바닥 높이":<12}{"대수":>6}  ' + ''.join(f'{f"{k} conf{c}":>19}' for k, c in keys9))
groups = [(f'{a:.2f}~{min(b, 1):.2f}', lambda y, a=a, b=b: a <= y < b) for a, b in zip(BINS8[:-1], BINS8[1:])]
groups += [('0.58 위(먼 차)', lambda y: y < 0.58), ('0.58 이상', lambda y: y >= 0.58), ('전체', lambda y: True)]
for gname, cond in groups:
    cells, n = [], 0
    for k in keys9:
        rs = [r for r in RES9[k][0] if cond(r[0])]
        n = len(rs)
        cells.append(f'{sum(r[2] for r in rs) / n:.2f} ({sum(r[2] for r in rs)}/{n})' if n else '-')
    print(f'  {gname:<12}{n:>6}  ' + ''.join(f'{c:>19}' for c in cells))
print(f'  {"오탐":<12}{"":>6}  ' + ''.join(f'{RES9[k][1]:>19}' for k in keys9))
print('\n보는 법: 0.58 위 행에서 "320+먼 영역"이 640에 가까우면 → 먼 영역만 한 번 더 보는 것으로 충분(연산 약 2배)')

## 10. 먼 영역 검출 오탐 분류

In [ ]:
# 10. 먼 영역 검출(9번)의 오탐은 뭔가 — 같은 차 중복 박스인지, 라벨 없는 먼 차인지, 진짜 헛것인지
# 필요: 8번 → 9번 셀을 이 세션에서 먼저 실행 (model9·merge9·CROP_X9 등 사용). GPU, 30초 정도.
#
# 왜: 9번 — 320+먼 영역의 먼 차 Recall은 640과 거의 같지만(0.70 vs 0.72) 오탐이 640보다 많다(conf 0.5: 329 vs 188).
# 방법: ① 먼 영역에서 더한 박스 중 오탐을 셋으로 나눈다
#          중복-검출: 전체 화면 검출과 조금이라도 겹침(IoU 0.1 이상 또는 작은 박스의 절반 이상이 포개짐)
#          중복-정답: 이미 찾은/못 찾은 정답 차와 겹침(같은 기준) — 박스가 차를 어긋나게 잡은 것
#          단독: 아무것과도 안 겹침 — 라벨 없는 먼 차이거나 헛것 → 그림으로 직접 본다
#       ② 더하는 규칙을 바꿔(IoU 0.5 미만 → 0.3 미만 → '포개짐' 기준) 오탐이 줄고 Recall은 유지되는지
import numpy as np, time
from PIL import Image, ImageDraw

if 'model9' not in globals():
    raise SystemExit('8번 → 9번 셀을 이 세션에서 먼저 실행하세요')

TOUCH10 = 0.1                                          # 이 이상 겹치면 '겹침'
GRID10  = 48                                           # 단독 오탐 그림 수


def inter10(b, p):
    """박스 b와 p 각각의 교집합 / 둘 중 작은 박스 넓이"""
    p = p.reshape(-1, 4)
    iw = np.clip(np.minimum(b[2], p[:, 2]) - np.maximum(b[0], p[:, 0]), 0, None)
    ih = np.clip(np.minimum(b[3], p[:, 3]) - np.maximum(b[1], p[:, 1]), 0, None)
    a = (b[2] - b[0]) * (b[3] - b[1]); ap = (p[:, 2] - p[:, 0]) * (p[:, 3] - p[:, 1])
    return iw * ih / np.maximum(np.minimum(a, ap), 1e-9)


def touch10(b, p):
    p = np.asarray(p).reshape(-1, 4)
    return len(p) > 0 and (iou8(b, p).max() >= TOUCH10 or inter10(b, p).max() >= 0.5)


def match10(g, p):
    """match9와 같은 짝짓기, 대신 쓰인 예측 번호도 돌려준다"""
    used, hits = set(), []
    for gb in sorted(g, key=lambda b: -(b[3] - b[1])):
        hit = False
        if len(p):
            ious = iou8(gb, p)
            for j in np.argsort(-ious):
                if ious[j] < IOU8:
                    break
                if j not in used:
                    used.add(j); hit = True
                    break
        hits.append((gb[3], hit))
    return hits, used


RULES10 = {
    'IoU<0.5 (9번)': lambda b, pf: len(pf) == 0 or iou8(b, pf).max() < 0.5,
    'IoU<0.3':       lambda b, pf: len(pf) == 0 or iou8(b, pf).max() < 0.3,
    '안 포개짐':      lambda b, pf: len(pf) == 0 or (iou8(b, pf).max() < 0.3 and inter10(b, pf).max() < 0.5),
}
STAT10 = {(r, c): dict(hits=[], fp_full=0, fp_add=0) for r in RULES10 for c in CONF8}
KIND10 = {c: {'중복-검출': 0, '중복-정답': 0, '단독': 0} for c in CONF8}
ALONE10 = []                                            # (이미지, 박스, conf) — conf 0.5 단독 오탐
cw, ch = CROP_X9[1] - CROP_X9[0], CROP_Y9[1] - CROP_Y9[0]
t0 = time.time()
for i in range(0, len(imgs8), 32):
    batch = imgs8[i:i + 32]
    rf = model9.predict([str(p) for p in batch], imgsz=320, conf=min(CONF8), verbose=False, device=0)
    crops = []
    for p in batch:
        im = Image.open(p); w, h = im.size
        crops.append(im.crop((int(CROP_X9[0] * w), int(CROP_Y9[0] * h), int(CROP_X9[1] * w), int(CROP_Y9[1] * h))))
    rc = model9.predict(crops, imgsz=320, conf=min(CONF8), verbose=False, device=0)
    for j, ip in enumerate(batch):
        g = GT8[ip]
        if len(g) == 0:
            continue
        pz, sz_ = rc[j].boxes.xyxyn.cpu().numpy(), rc[j].boxes.conf.cpu().numpy()
        inner = ((pz[:, 0] > EDGE9 / cw) & (pz[:, 2] < 1 - EDGE9 / cw) & (pz[:, 1] > EDGE9 / ch) & (pz[:, 3] < 1 - EDGE9 / ch)) if len(pz) else np.zeros(0, bool)
        pz, sz_ = pz[inner], sz_[inner]
        pz = np.c_[CROP_X9[0] + pz[:, 0] * cw, CROP_Y9[0] + pz[:, 1] * ch, CROP_X9[0] + pz[:, 2] * cw, CROP_Y9[0] + pz[:, 3] * ch] if len(pz) else pz.reshape(0, 4)
        pf_all, sf = rf[j].boxes.xyxyn.cpu().numpy(), rf[j].boxes.conf.cpu().numpy()
        for c in CONF8:
            pf = pf_all[sf >= c]
            for r, keep in RULES10.items():
                add = [(b, s) for b, s in zip(pz[sz_ >= c], sz_[sz_ >= c]) if keep(b, pf)]
                q = np.r_[pf.reshape(-1, 4), np.array([b for b, _ in add]).reshape(-1, 4)]
                hits, used = match10(g, q)
                st = STAT10[(r, c)]
                st['hits'] += hits
                for k in range(len(q)):
                    if k in used or abs((q[k, 0] + q[k, 2]) / 2 - 0.5) > CENTER8 / 2:
                        continue
                    if k < len(pf):
                        st['fp_full'] += 1
                        continue
                    st['fp_add'] += 1
                    if r != 'IoU<0.5 (9번)':
                        continue
                    b = q[k]
                    kind = '중복-검출' if touch10(b, pf) else '중복-정답' if touch10(b, g) else '단독'
                    KIND10[c][kind] += 1
                    if kind == '단독' and c == max(CONF8):
                        ALONE10.append((ip, b, float(add[k - len(pf)][1])))
print(f'{len(imgs8)}장 {time.time() - t0:.0f}초')

print('\n① 먼 영역에서 더한 박스의 오탐 내역 (9번 규칙)')
for c in CONF8:
    print(f'  conf {c}: ' + ' · '.join(f'{k} {v}' for k, v in KIND10[c].items()))

print('\n② 더하는 규칙별 — 먼 차(바닥 0.58 위) Recall · 0.58 이상 Recall · 오탐(전체 화면 / 먼 영역에서 더한 것)')
print(f'  {"규칙":<16}' + ''.join(f'{f"conf {c}":>44}' for c in CONF8))
for r in RULES10:
    line = f'  {r:<16}'
    for c in CONF8:
        st = STAT10[(r, c)]
        far = [h for y, h in st['hits'] if y < 0.58]; near = [h for y, h in st['hits'] if y >= 0.58]
        line += f'{np.mean(far):>12.2f}{np.mean(near):>8.2f}{st["fp_full"]:>10} / {st["fp_add"]:<8}'
    print(line)
print('  참고(9번) 640: 먼 차 0.86 / 0.72, 오탐 1230 / 188')

ALONE10.sort(key=lambda t: -t[2])
tiles = []
for ip, b, s in ALONE10[:GRID10]:
    im = Image.open(ip).convert('RGB'); w, h = im.size
    x1, y1, x2, y2 = b[0] * w, b[1] * h, b[2] * w, b[3] * h
    m = max(x2 - x1, y2 - y1) * 1.5
    box = (int(max(0, x1 - m)), int(max(0, y1 - m)), int(min(w, x2 + m)), int(min(h, y2 + m)))
    t = im.crop(box); d = ImageDraw.Draw(t)
    for gb in GT8[ip]:
        d.rectangle([gb[0] * w - box[0], gb[1] * h - box[1], gb[2] * w - box[0], gb[3] * h - box[1]], outline=(0, 255, 0), width=2)
    d.rectangle([x1 - box[0], y1 - box[1], x2 - box[0], y2 - box[1]], outline=(255, 0, 0), width=2)
    d.text((3, 3), f'{s:.2f}', fill=(255, 255, 0))
    tiles.append(t.resize((160, 160)))
if tiles:
    cols = 8
    sheet = Image.new('RGB', (cols * 160, ((len(tiles) - 1) // cols + 1) * 160))
    for k, t in enumerate(tiles):
        sheet.paste(t, ((k % cols) * 160, (k // cols) * 160))
    sheet.save('/content/fp10.jpg', quality=90)
    display(sheet)
    print(f'conf 0.5 단독 오탐 {len(ALONE10)}개 중 확신 높은 순 {len(tiles)}개 — 빨강 = 먼 영역 검출, 초록 = 정답. /content/fp10.jpg')
print('\n보는 법: ①에서 중복이 대부분이면 → ② 규칙만 바꾸면 해결. 단독이 많으면 그림에서 차가 보이는지(라벨 누락) 확인')


---## 재개 (학습 중 세션이 끊긴 경우)1 → 2번 셀을 실행한 뒤 아래를 실행한다. 중단된 에폭부터 이어서 학습한다.

In [ ]:
last = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
assert last.exists(), f'체크포인트 없음: {last}'

model = YOLO(str(last))
model.train(resume=True)